In [2]:
# ipynb 테스트 코드에 추가하여 검증
overall_monthly = data_bundle['전체']['monthly']
total_hires = overall_monthly['NEW_HIRES'].sum()
total_leavers = overall_monthly['LEAVERS'].sum()

print(f"전체 기간 총 입사자 수: {total_hires}")
print(f"전체 기간 총 퇴사자 수: {total_leavers}")

전체 기간 총 입사자 수: 1000
전체 기간 총 퇴사자 수: 520


In [4]:
# 2. 필요한 모듈 임포트
import pandas as pd
import plotly.io as pio
from IPython.display import display

from services.data_preparer import prepare_basic_proposal_data
from services.proposal_views.basic_proposal_view import create_figure_and_df

# --- 테스트 실행 ---

# 1. app.py의 역할 (1): 데이터 준비 함수를 호출하여 필요한 모든 데이터를 미리 로드합니다.
print("Step 1: `data_preparer`를 통해 데이터 묶음(bundle)과 순서 정보를 준비합니다...")
data_bundle_full = prepare_basic_proposal_data()
data_bundle = data_bundle_full.get("data_bundle", {})
order_map = data_bundle_full.get("order_map", {})
print(" -> 데이터 준비 완료!")


# 2. app.py의 역할 (2): 사용자가 Streamlit 위젯을 통해 필터를 선택했다고 가정합니다.

# 2-1. app.py에 있을 차원 설정(Dimension Config) 정의
# 이 Config는 view 함수가 동적으로 작동하기 위한 '설명서' 역할을 합니다.
DIMENSION_CONFIG = {
    '부서별': {'type': 'hierarchical', 'top': 'DIVISION_NAME', 'sub': 'OFFICE_NAME'},
    '직무별': {'type': 'hierarchical', 'top': 'JOB_L1_NAME', 'sub': 'JOB_L2_NAME'},
    '직위직급별': {'type': 'flat', 'col': 'POSITION_NAME'},
    '성별': {'type': 'flat', 'col': 'GENDER'},
    '연령별': {'type': 'flat', 'col': 'AGE_BIN'},
    '경력연차별': {'type': 'flat', 'col': 'CAREER_BIN'},
    '연봉구간별': {'type': 'flat', 'col': 'SALARY_BIN'},
    '지역별': {'type': 'flat', 'col': 'REGION_CATEGORY'},
    '계약별': {'type': 'flat', 'col': 'CONT_CATEGORY'}
}
# order_map의 순서 정보를 config에 추가
for k, v in DIMENSION_CONFIG.items():
    if v['type'] == 'hierarchical':
        v['order'] = order_map.get(v['top'])
        v['sub_order'] = order_map.get(v['sub'])
    else: # flat
        v['order'] = order_map.get(v['col'])

# 2-2. 사용자 선택 시뮬레이션
# ----- 테스트하고 싶은 값으로 변경 -----
selected_dimension_ui = '성별'
selected_period_agg = 'yearly'
selected_subgroup = '남성' # '부서별' 선택 시 'Development Division' 등으로 변경 가능
# -----------------------------------

print(f"Step 2: 사용자가 '{selected_dimension_ui}', '{selected_period_agg.replace('ly','별')}', '{selected_subgroup}' 필터를 선택했습니다.")


# 3. app.py의 역할 (3): view 함수에 준비된 모든 데이터와 필터 값을 전달하여 결과물 생성
print("Step 3: `view` 모듈을 호출하여 그래프와 요약 테이블을 생성합니다...")
if data_bundle:
    fig, aggregate_df = create_figure_and_df(
        data_bundle=data_bundle, 
        dimension_ui_name=selected_dimension_ui, 
        period_agg_name=selected_period_agg,
        subgroup_name=selected_subgroup,
        dimension_config=DIMENSION_CONFIG
    )
    print(" -> 생성 완료!")
else:
    print(" -> 데이터 번들이 비어있어 결과물을 생성할 수 없습니다.")
    fig, aggregate_df = go.Figure(), pd.DataFrame()


# --- 결과 확인 ---

# 4. ipynb에서 생성된 그래프를 확인합니다.
print("\n--- [결과 1] 생성된 Plotly 그래프 ---")
# pio.renderers.default = 'vscode' 
fig.show()

# 5. ipynb에서 생성된 요약 테이블(aggregate_df)을 확인합니다.
print(f"\n--- [결과 2] '{selected_dimension_ui}' 기준 생성된 요약 테이블 ---")
display(aggregate_df)

Step 1: `data_preparer`를 통해 데이터 묶음(bundle)과 순서 정보를 준비합니다...
 -> 데이터 준비 완료!
Step 2: 사용자가 '성별', 'year별', '남성' 필터를 선택했습니다.
Step 3: `view` 모듈을 호출하여 그래프와 요약 테이블을 생성합니다...
 -> 생성 완료!

--- [결과 1] 생성된 Plotly 그래프 ---



--- [결과 2] '성별' 기준 생성된 요약 테이블 ---


,전체,남성,여성
PERIOD,,,
2014년,244,130,114
2015년,301,161,140
2016년,329,170,159
2017년,365,188,177
2018년,397,204,193
2019년,425,219,206
2020년,457,226,231
2021년,471,238,233
2022년,488,244,244
